# Oil Spill Semantic Segmentation: U-Net track

**How to run on Kaggle:** add the dataset `nabilsherif/oil-spill` to the notebook, enable a GPU, then *Save Version -> Save & Run All*. Outputs (weights, logs, plots, `results/`) are written to `/kaggle/working`. Checkpoints are saved every epoch (`last.pth`), so a stopped run can resume.

Order: config -> code cells (from the project modules) -> audit -> experiments -> curves -> evaluation -> table. Generated by `build_notebook.py`; do not edit code here, edit the `.py` files and rebuild.

In [ ]:
import os, sys, platform, json
ON_KAGGLE = os.path.exists("/kaggle/input")
DATA_ROOT = os.environ.get("OIL_DATA_ROOT") or ("/kaggle/input" if ON_KAGGLE else None)   # folder that contains train/ and test/
OUT = os.environ.get("OIL_OUT") or ("/kaggle/working" if ON_KAGGLE else "nb_out")        # everything is saved here
EPOCHS = int(os.environ.get("OIL_EPOCHS", 40))
H, W = int(os.environ.get("OIL_H", 256)), int(os.environ.get("OIL_W", 512))                # images are ~2:1 wide
BASE = int(os.environ.get("OIL_BASE", 32))
WORKERS = 2 if ON_KAGGLE else 0
RUN_ABLATIONS = os.environ.get("OIL_ABLATIONS", "0") == "1"   # also run weights-only and Dice-only runs
FINAL = os.environ.get("OIL_FINAL", "0") == "1"                # set True ONCE, after every experiment/checkpoint choice is frozen
assert DATA_ROOT, "Set DATA_ROOT to the folder containing train/ and test/"
os.makedirs(OUT, exist_ok=True); os.chdir(OUT)
import torch
print("python", platform.python_version(), "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("DATA_ROOT =", DATA_ROOT, "| OUT =", os.getcwd(), "| input", H, "x", W, "| epochs", EPOCHS)

In [ ]:
from IPython.display import Image as _Img, display
def show(*paths):
    for p in paths:
        if os.path.exists(p):
            display(_Img(filename=p))

## Data pipeline (shared contract)
`oilspill/data.py`

In [ ]:
"""Shared data pipeline for the Oil Spill segmentation assignment.

This module is the "shared contract" Omar and Abdallah both import, so that the
class mapping, split and resizing are identical for both models. Normalisation is a
PARAMETER because it must match the model: "half" for the from-scratch U-Net,
"imagenet" for a pretrained DeepLabV3+ backbone.
"""
import json
import random
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset

CLASS_NAMES = ["sea", "oil_spill", "look_alike", "ship", "land"]
# Colours as stated in the assignment. audit_dataset.py checks them against the real masks.
DEFAULT_COLORS = [(0, 0, 0), (0, 255, 255), (255, 0, 0), (153, 76, 0), (0, 153, 0)]
IMG_EXT = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}
NORMS = {  # (mean, std) per channel applied to x/255
    "half": ((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),                      # -> [-1, 1]; scratch U-Net
    "imagenet": ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),      # pretrained backbones
}


def load_colors(class_map_path=None):
    if class_map_path and Path(class_map_path).exists():
        return [tuple(c) for c in json.load(open(class_map_path))["colors"]]
    return DEFAULT_COLORS


def rgb_to_label(mask_rgb, colors):
    """Map an HxWx3 colour mask to HxW integer labels via the nearest palette colour.

    Nearest-colour (not exact match) so slightly off colours from compression do not
    silently become class 0. audit_dataset.py reports how many pixels are not exact.
    """
    m = mask_rgb.astype(np.int32)[..., None, :]  # int32: 255**2 overflows int16
    pal = np.asarray(colors, dtype=np.int32)[None, None]
    return ((m - pal) ** 2).sum(-1).argmin(-1).astype(np.uint8)


def load_mask_labels(path, size_hw, colors):
    """Load a mask, resize with NEAREST (never interpolate labels), return HxW uint8 labels."""
    im = Image.open(path)
    if size_hw is not None:
        im = im.resize((size_hw[1], size_hw[0]), Image.NEAREST)  # PIL takes (w, h)
    if im.mode in ("L", "P", "I", "I;16"):  # already single-channel label / palette indices
        return np.asarray(im).astype(np.uint8)
    return rgb_to_label(np.asarray(im.convert("RGB")), colors)


def _stem_map(directory):
    return {p.stem: p for p in Path(directory).iterdir() if p.suffix.lower() in IMG_EXT}


def find_pairs(img_dir, mask_dir, prefix=""):
    """Pair images with masks by file stem. `prefix` namespaces the returned IDs: the real dataset numbers
    BOTH train and test from img_0001, so test pairs must use prefix="test_" or the split IDs collide."""
    imgs, masks = _stem_map(img_dir), _stem_map(mask_dir)
    ids = sorted(set(imgs) & set(masks))
    return [(prefix + i, imgs[i], masks[i]) for i in ids], sorted(set(imgs) - set(masks)), sorted(set(masks) - set(imgs))


def discover(data_root):
    """Find {split: (image_dir, mask_dir)} under data_root without assuming exact folder names.

    Looks for train/test folders containing an 'image*' dir and a 'label*'/'mask*' dir.
    Prefers RGB label folders over '*1D*' (single-channel) ones.
    """
    root = Path(data_root)
    found = {}
    for split in ("train", "test"):
        for d in root.rglob("*"):
            if not d.is_dir() or d.name.lower() != split:
                continue
            subs = [s for s in d.iterdir() if s.is_dir()]
            imgs = [s for s in subs if "image" in s.name.lower()]
            lab = [s for s in subs if any(k in s.name.lower() for k in ("label", "mask"))]
            lab.sort(key=lambda s: ("1d" in s.name.lower(), s.name))
            if imgs and lab:
                found[split] = (imgs[0], lab[0])
                break
    return found


def make_split(train_ids, val_frac=0.15, seed=42):
    ids = sorted(train_ids)
    random.Random(seed).shuffle(ids)
    n_val = int(round(len(ids) * val_frac))
    return sorted(ids[n_val:]), sorted(ids[:n_val])


def load_or_make_split(split_path, train_ids, test_ids, val_frac=0.15, seed=42):
    """Reproducible train/val split stored in a JSON manifest the whole team shares."""
    split_path = Path(split_path)
    if split_path.exists():
        s = json.load(open(split_path))
        assert set(s["train"]) | set(s["val"]) == set(train_ids), "splits.json does not match dataset"
        return s
    tr, va = make_split(train_ids, val_frac, seed)
    assert not (set(tr) | set(va)) & set(test_ids), "train/val/test leakage"
    s = {"seed": seed, "val_frac": val_frac, "train": tr, "val": va, "test": sorted(test_ids)}
    split_path.parent.mkdir(parents=True, exist_ok=True)
    json.dump(s, open(split_path, "w"), indent=1)
    return s


class OilSpillDataset(Dataset):
    """size_hw=(H, W). The real images are ~2:1 wide, so a non-square size avoids squashing.

    augment: False/"none" | "flip" (h+v flips) | "flip_int" (flips + mild brightness/contrast
    jitter on the IMAGE only; labels never get intensity changes).
    """

    def __init__(self, items, size_hw=(256, 512), augment="none", colors=None, norm="half", return_id=False):
        self.items, self.size_hw = items, tuple(size_hw)
        self.augment = "flip" if augment is True else (augment or "none")
        self.colors, self.return_id = colors or DEFAULT_COLORS, return_id
        mean, std = NORMS[norm]
        self.mean = torch.tensor(mean).view(3, 1, 1)
        self.std = torch.tensor(std).view(3, 1, 1)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        sid, ip, mp = self.items[i]
        h, w = self.size_hw
        img = Image.open(ip).convert("RGB").resize((w, h), Image.BILINEAR)
        x = torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1).float() / 255
        y = torch.from_numpy(load_mask_labels(mp, self.size_hw, self.colors).astype(np.int64))
        if self.augment in ("flip", "flip_int"):  # identical spatial transform on image and mask
            if random.random() < 0.5:
                x, y = x.flip(-1), y.flip(-1)
            if random.random() < 0.5:
                x, y = x.flip(-2), y.flip(-2)
        if self.augment == "flip_int":  # image only
            x = ((x - 0.5) * random.uniform(0.8, 1.2) + 0.5 + random.uniform(-0.1, 0.1)).clamp(0, 1)
        x = (x - self.mean) / self.std
        return (x, y, sid) if self.return_id else (x, y)


def class_pixel_counts(items, size_hw, colors, n_classes=5):
    counts = np.zeros(n_classes, dtype=np.int64)
    for _, _, mp in items:
        counts += np.bincount(load_mask_labels(mp, size_hw, colors).ravel(), minlength=n_classes)[:n_classes]
    return counts


def select(pairs, ids):
    wanted = set(ids)
    return [p for p in pairs if p[0] in wanted]

## U-Net from scratch
`oilspill/unet.py`

In [ ]:
"""U-Net implemented from scratch (no pretrained weights, no segmentation library)."""
import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class Down(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.pool, self.conv = nn.MaxPool2d(2), DoubleConv(cin, cout)

    def forward(self, x):
        return self.conv(self.pool(x))


class Up(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.up = nn.ConvTranspose2d(cin, cin // 2, 2, stride=2)
        self.conv = DoubleConv(cin, cout)  # cin//2 (upsampled) + cin//2 (skip) = cin

    def forward(self, x, skip):
        x = self.up(x)
        dy, dx = skip.size(2) - x.size(2), skip.size(3) - x.size(3)
        x = F.pad(x, [dx // 2, dx - dx // 2, dy // 2, dy - dy // 2])  # odd input sizes
        return self.conv(torch.cat([skip, x], dim=1))


class UNet(nn.Module):
    """encoder -> bottleneck -> decoder with skip connections -> 1x1 conv to n_classes logits."""

    def __init__(self, in_ch=3, n_classes=5, base=32, depth=4):
        super().__init__()
        chs = [base * 2 ** i for i in range(depth + 1)]  # e.g. 32,64,128,256,512
        self.inc = DoubleConv(in_ch, chs[0])
        self.downs = nn.ModuleList([Down(chs[i], chs[i + 1]) for i in range(depth)])  # last = bottleneck
        self.ups = nn.ModuleList([Up(chs[i + 1], chs[i]) for i in reversed(range(depth))])
        self.head = nn.Conv2d(chs[0], n_classes, 1)

    def forward(self, x):
        skips = [self.inc(x)]
        for d in self.downs:
            skips.append(d(skips[-1]))
        x = skips.pop()  # bottleneck output
        for up in self.ups:
            x = up(x, skips.pop())
        return self.head(x)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Losses
`oilspill/losses.py`

In [ ]:
"""Losses. Four kinds so the U2 change can be decomposed into its parts:

    ce        plain cross-entropy                       (U1 baseline)
    wce       class-weighted cross-entropy              (isolates the effect of weights)
    ce_dice   cross-entropy + soft Dice, no weights     (isolates the effect of Dice)
    wce_dice  weighted cross-entropy + soft Dice        (U2: both together)
"""
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

NEEDS_WEIGHTS = ("wce", "wce_dice")


def class_weights_from_counts(counts, method="inv_sqrt"):
    """Weights from TRAINING-split pixel counts only (never val/test).

    inv_sqrt:    w_c ~ 1/sqrt(freq_c)  (milder, usually more stable)
    median_freq: w_c = median(freq) / freq_c  (Eigen & Fergus style, stronger)
    Normalised to mean 1 so the loss scale stays comparable to plain CE.
    """
    freq = np.asarray(counts, dtype=np.float64)
    freq = freq / freq.sum()
    freq = np.maximum(freq, 1e-8)  # a class absent from train would otherwise give inf
    w = 1.0 / np.sqrt(freq) if method == "inv_sqrt" else np.median(freq) / freq
    return torch.tensor(w / w.mean(), dtype=torch.float32)


class SoftDiceLoss(nn.Module):
    """1 - mean per-class soft Dice over the batch. eps smooths classes absent from a batch."""

    def __init__(self, n_classes=5, eps=1.0):
        super().__init__()
        self.n, self.eps = n_classes, eps

    def forward(self, logits, target):
        p = F.softmax(logits.float(), dim=1)
        t = F.one_hot(target, self.n).permute(0, 3, 1, 2).float()
        inter = (p * t).sum((0, 2, 3))
        denom = p.sum((0, 2, 3)) + t.sum((0, 2, 3))
        return 1 - ((2 * inter + self.eps) / (denom + self.eps)).mean()


class CEDice(nn.Module):
    def __init__(self, weight=None, dice_w=1.0, n_classes=5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=weight)
        self.dice, self.dice_w = SoftDiceLoss(n_classes), dice_w

    def forward(self, logits, target):
        return self.ce(logits.float(), target) + self.dice_w * self.dice(logits, target)


def build_loss(kind, weights=None, dice_w=1.0):
    if kind in NEEDS_WEIGHTS and weights is None:
        raise ValueError(f"loss '{kind}' needs class weights")
    if kind == "ce":
        return nn.CrossEntropyLoss()
    if kind == "wce":
        return nn.CrossEntropyLoss(weight=weights)
    if kind == "ce_dice":
        return CEDice(weight=None, dice_w=dice_w)
    if kind == "wce_dice":
        return CEDice(weight=weights, dice_w=dice_w)
    raise ValueError(kind)

## Metrics
`oilspill/metrics.py`

In [ ]:
"""Segmentation metrics from a confusion matrix (rows = ground truth, cols = prediction).

Absent-class policy: a class with no ground-truth AND no predicted pixels has an undefined
IoU/Dice (0/0). It is reported as null and EXCLUDED from the mIoU mean (nanmean), rather
than counted as 0 or 1. Say this in the report.
"""
import numpy as np
import torch

NAMES = ["sea", "oil_spill", "look_alike", "ship", "land"]


def confusion(pred, target, n=5):
    idx = target.reshape(-1) * n + pred.reshape(-1)
    return torch.bincount(idx, minlength=n * n).reshape(n, n)


def summarize(cm, names=NAMES):
    cm = np.asarray(cm, dtype=np.float64)
    tp = np.diag(cm)
    fp, fn = cm.sum(0) - tp, cm.sum(1) - tp
    with np.errstate(invalid="ignore", divide="ignore"):
        iou = np.where(tp + fp + fn > 0, tp / (tp + fp + fn), np.nan)
        dice = np.where(2 * tp + fp + fn > 0, 2 * tp / (2 * tp + fp + fn), np.nan)
    f = lambda v: None if np.isnan(v) else float(v)
    out = {"mIoU": f(np.nanmean(iou)), "pixel_acc": float(tp.sum() / max(cm.sum(), 1))}
    for i, n in enumerate(names):
        out[f"IoU_{n}"], out[f"Dice_{n}"] = f(iou[i]), f(dice[i])
    out["mDice"] = f(np.nanmean(dice))
    return out

## Visualisation helpers
`oilspill/viz.py`

In [ ]:
import numpy as np



def colorize(label, colors=None):
    pal = np.asarray(colors or DEFAULT_COLORS, dtype=np.uint8)
    return pal[np.asarray(label)]


def denorm(x, norm="half"):
    mean, std = NORMS[norm]
    x = x.permute(1, 2, 0).cpu().numpy() * np.asarray(std) + np.asarray(mean)
    return np.clip(x, 0, 1)

## Dataset audit
`audit_dataset.py`

In [ ]:
"""Phase U0: audit the dataset. Run this FIRST on the real data.

    python audit_dataset.py --data_root <folder with train/ and test/> --out audit

Checks folder structure, image/mask alignment, image sizes, whether mask colours really match
the colours the assignment states, class frequencies, how many images contain each class, and
writes a sample grid. It writes class_map.json (colours + names) which every other script uses.
(Looking at test masks here is inspection only: nothing is tuned on them.)
"""
import argparse
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image



def audit_parser():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_root", required=True)
    ap.add_argument("--out", default="audit")
    ap.add_argument("--max_masks", type=int, default=0, help="0 = scan every training mask")
    return ap


def run_audit(a):
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)

    found = discover(a.data_root)
    print("Discovered:", {k: (str(v[0]), str(v[1])) for k, v in found.items()})
    assert "train" in found, "No train/ folder with images + labels found; check --data_root"
    report = {"data_root": str(a.data_root), "splits": {}}
    pairs = {}
    for split, (idir, mdir) in found.items():
        p, no_mask, no_img = find_pairs(idir, mdir)
        pairs[split] = p
        report["splits"][split] = {"images_dir": str(idir), "masks_dir": str(mdir), "pairs": len(p),
                                   "images_without_mask": no_mask, "masks_without_image": no_img}
        print(f"[{split}] pairs={len(p)} imgs_without_mask={len(no_mask)} masks_without_img={len(no_img)}")

    # leakage check by CONTENT (names are not enough: train and test both start at img_0001)
    if "test" in pairs:
        import hashlib
        md5 = lambda p: hashlib.md5(Path(p).read_bytes()).hexdigest()
        train_h = {md5(ip): sid for sid, ip, _ in pairs["train"]}
        dups = [(sid, train_h[md5(ip)]) for sid, ip, _ in pairs["test"] if md5(ip) in train_h]
        shared = len({p[0] for p in pairs["train"]} & {p[0] for p in pairs["test"]})
        report["train_test_identical_images"] = dups
        report["train_test_shared_file_names"] = shared
        print(f"train/test leakage: {len(dups)} test images are byte-identical to a train image"
              f" | {shared} file names are reused across the two folders (harmless; test IDs get a 'test_' prefix)")

    # sizes / modes + image-mask alignment
    sizes, modes, mismatch = Counter(), Counter(), []
    for _, ip, mp in pairs["train"]:
        with Image.open(ip) as im, Image.open(mp) as mk:
            sizes[im.size] += 1
            modes[(im.mode, mk.mode)] += 1
            if im.size != mk.size:
                mismatch.append(ip.stem)
    report["image_sizes"] = {f"{w}x{h}": n for (w, h), n in sizes.most_common()}
    report["(image_mode, mask_mode)"] = {str(k): n for k, n in modes.items()}
    report["size_mismatch_image_vs_mask"] = mismatch
    top = list(report["image_sizes"].items())[:5]
    print(f"image sizes (w x h, top 5 of {len(sizes)}):", top, "| modes:", report["(image_mode, mask_mode)"])
    print("image/mask size mismatches:", len(mismatch))

    # mask colour verification on training masks (native resolution, no resizing)
    items = pairs["train"][: a.max_masks or None]
    colour_counts, label_values = Counter(), Counter()
    for _, _, mp in items:
        m = Image.open(mp)
        if m.mode in ("L", "P", "I", "I;16"):
            u, c = np.unique(np.asarray(m), return_counts=True)
            label_values.update(dict(zip(map(int, u), map(int, c))))
        else:
            arr = np.asarray(m.convert("RGB")).reshape(-1, 3).astype(np.int32)
            packed = (arr[:, 0] << 16) | (arr[:, 1] << 8) | arr[:, 2]  # 1-D unique is far faster than axis=0
            u, c = np.unique(packed, return_counts=True)
            for p, n in zip(u.tolist(), c):
                colour_counts[(p >> 16, (p >> 8) & 255, p & 255)] += int(n)
    if colour_counts:
        total = sum(colour_counts.values())
        exact = sum(colour_counts.get(tuple(c), 0) for c in DEFAULT_COLORS)
        print(f"\nTop mask colours (of {len(colour_counts)} distinct):")
        for col, n in colour_counts.most_common(10):
            tag = CLASS_NAMES[DEFAULT_COLORS.index(col)] if col in DEFAULT_COLORS else "NOT IN ASSIGNMENT"
            print(f"  {col!s:<18} {n / total:7.3%}  {tag}")
        report["mask_exact_match_fraction"] = exact / total
        report["distinct_mask_colours"] = len(colour_counts)
        report["top_colours"] = [{"rgb": list(c), "fraction": n / total} for c, n in colour_counts.most_common(10)]
        print(f"exact assignment-colour match: {exact / total:.3%}"
              + ("  <-- <99%: masks have extra/off colours (JPEG?). Nearest-colour mapping is used; read the table above."
                 if exact / total < 0.99 else ""))
    if label_values:
        print("single-channel label values:", dict(sorted(label_values.items())))
        report["label_values"] = dict(sorted(label_values.items()))

    json.dump({"names": CLASS_NAMES, "colors": [list(c) for c in DEFAULT_COLORS],
               "verified_exact_match_fraction": report.get("mask_exact_match_fraction")},
              open(out / "class_map.json", "w"), indent=1)

    # pixel share (train) + number of images containing each class (every split)
    counts = np.zeros(5, dtype=np.int64)
    present = {s: np.zeros(5, dtype=np.int64) for s in pairs}
    for s, plist in pairs.items():
        for _, _, mp in (items if s == "train" else plist):
            b = np.bincount(load_mask_labels(mp, None, DEFAULT_COLORS).ravel(), minlength=5)[:5]
            present[s] += (b > 0)
            if s == "train":
                counts += b
    freq = counts / counts.sum()
    report["train_class_pixel_share"] = {n: float(f) for n, f in zip(CLASS_NAMES, freq)}
    report["images_containing_class"] = {s: {n: int(v) for n, v in zip(CLASS_NAMES, c)} for s, c in present.items()}
    print("\ntrain class share:", {n: f"{f:.3%}" for n, f in zip(CLASS_NAMES, freq)})
    for s, c in present.items():
        n_img = len(items) if s == "train" else len(pairs[s])
        print(f"images containing each class [{s}, of {n_img}]:", {n: int(v) for n, v in zip(CLASS_NAMES, c)})
        if (c == 0).any():
            print(f"  <-- WARNING: class(es) {[CLASS_NAMES[i] for i in np.where(c == 0)[0]]} absent from the {s} split")
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.bar(CLASS_NAMES, freq, color=[np.array(c) / 255 for c in DEFAULT_COLORS], edgecolor="k")
    ax.set_yscale("log"); ax.set_ylabel("pixel share (log)"); ax.set_title("Training-set class frequency")
    fig.tight_layout(); fig.savefig(out / "class_frequency.png", dpi=150); plt.close(fig)

    # sample grid: image | mask
    pick = items[:: max(1, len(items) // 6)][:6]
    fig, axs = plt.subplots(2, len(pick), figsize=(3 * len(pick), 4.5))
    for j, (sid, ip, mp) in enumerate(pick):
        axs[0, j].imshow(Image.open(ip).convert("RGB")); axs[0, j].set_title(sid, fontsize=8)
        axs[1, j].imshow(colorize(load_mask_labels(mp, None, DEFAULT_COLORS)))
        for r in (0, 1):
            axs[r, j].axis("off")
    fig.tight_layout(); fig.savefig(out / "sample_grid.png", dpi=130); plt.close(fig)

    json.dump(report, open(out / "audit_report.json", "w"), indent=1)
    print(f"\nWrote {out}/audit_report.json, class_map.json, class_frequency.png, sample_grid.png")

## Training
`train.py`

In [ ]:
"""Train one U-Net experiment. Never touches the test set.

    U1 baseline            python train.py --data_root D --run_id U1  --loss ce
    U2 weights + Dice      python train.py --data_root D --run_id U2  --loss wce_dice
    ablations (optional)   --loss wce   (weights only)    --loss ce_dice  (Dice only)

Keep --seed/--height/--width/--bs/--epochs/--lr/--aug identical inside a controlled pair so only
the thing you are studying changes. Outputs (shared result contract) go to results/<run_id>/.
last.pth is saved every epoch; add --resume to continue a stopped run.
"""
import argparse
import csv
import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader



def train_parser():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_root", required=True)
    ap.add_argument("--run_id", required=True)
    ap.add_argument("--loss", choices=["ce", "wce", "ce_dice", "wce_dice"], default="ce")
    ap.add_argument("--weight_method", choices=["inv_sqrt", "median_freq"], default="inv_sqrt")
    ap.add_argument("--dice_w", type=float, default=1.0)
    ap.add_argument("--aug", choices=["none", "flip", "flip_int"], default="flip")
    ap.add_argument("--norm", choices=list(NORMS), default="half")
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--bs", type=int, default=8)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--wd", type=float, default=1e-4)
    ap.add_argument("--height", type=int, default=256)
    ap.add_argument("--width", type=int, default=512, help="images are ~2:1 wide; avoid squashing to a square")
    ap.add_argument("--base", type=int, default=32)
    ap.add_argument("--depth", type=int, default=4)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--val_frac", type=float, default=0.15)
    ap.add_argument("--workers", type=int, default=0)
    ap.add_argument("--no_amp", action="store_true")
    ap.add_argument("--deterministic", action="store_true", help="slower; makes runs repeatable on the same GPU")
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--class_map", default="audit/class_map.json")
    ap.add_argument("--splits", default="splits.json")
    ap.add_argument("--out", default="results")
    return ap


def seed_all(s, deterministic=False):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    if deterministic:
        torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False
        torch.use_deterministic_algorithms(True, warn_only=True)


def run_epoch(model, loader, crit, dev, opt=None, scaler=None, amp=False):
    train = opt is not None
    model.train(train)
    cm = torch.zeros(5, 5, dtype=torch.long, device=dev)
    loss_sum, ce_sum, n = 0.0, 0.0, 0
    ce_plain = torch.nn.CrossEntropyLoss()
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                logits = model(x)
            logits = logits.float()  # loss in fp32: fp16 logits overflow plain CE under AMP
            loss = crit(logits, y)
            if train:
                opt.zero_grad(set_to_none=True)
                if scaler:
                    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                else:
                    loss.backward(); opt.step()
            b = x.size(0)
            loss_sum += loss.item() * b
            ce_sum += ce_plain(logits, y).item() * b  # unweighted CE: comparable across runs
            n += b
            cm += confusion(logits.argmax(1), y)
    return loss_sum / n, ce_sum / n, summarize(cm.cpu().numpy())


COLS = ["run_id", "epoch", "train_loss", "val_loss", "train_ce", "val_ce", "train_mIoU", "val_mIoU",
        "val_oil_IoU", "val_ship_IoU", "val_lookalike_IoU", "epoch_time_s"]


def run_train(a):
    seed_all(a.seed, a.deterministic)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = dev.type == "cuda" and not a.no_amp
    size_hw = (a.height, a.width)
    colors = load_colors(a.class_map)
    found = discover(a.data_root)
    train_pairs = find_pairs(*found["train"])[0]
    test_ids = [p[0] for p in find_pairs(*found["test"], prefix="test_")[0]] if "test" in found else []
    split = load_or_make_split(a.splits, [p[0] for p in train_pairs], test_ids, a.val_frac, a.seed)
    tr_items, va_items = select(train_pairs, split["train"]), select(train_pairs, split["val"])
    print(f"[{a.run_id}] train={len(tr_items)} val={len(va_items)} (test={len(test_ids)} untouched) "
          f"device={dev} amp={amp} input={a.height}x{a.width} norm={a.norm} aug={a.aug} loss={a.loss}")

    weights = None
    if a.loss in NEEDS_WEIGHTS:  # weights from the TRAIN split only
        counts = class_pixel_counts(tr_items, size_hw, colors)
        weights = class_weights_from_counts(counts, a.weight_method)
        print("train pixel counts:", counts.tolist(), "-> class weights:", [round(float(w), 3) for w in weights])
        weights = weights.to(dev)
    crit = build_loss(a.loss, weights, a.dice_w)

    def loader(items, aug, shuffle):
        ds = OilSpillDataset(items, size_hw, aug, colors, a.norm)
        return DataLoader(ds, batch_size=a.bs, shuffle=shuffle, num_workers=a.workers, pin_memory=dev.type == "cuda")
    dl_tr, dl_va = loader(tr_items, a.aug, True), loader(va_items, "none", False)

    model = UNet(3, 5, a.base, a.depth).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=a.epochs)
    scaler = torch.amp.GradScaler("cuda") if amp else None

    out = Path(a.out) / a.run_id
    out.mkdir(parents=True, exist_ok=True)
    config = {**vars(a), "n_params": count_params(model), "class_weights": None if weights is None else weights.tolist(),
              "norm_mean_std": NORMS[a.norm], "colors": [list(c) for c in colors], "pretrained": False,
              "optimizer": "AdamW", "scheduler": "CosineAnnealingLR", "torch": torch.__version__,
              "augmentation": {"none": "none", "flip": "random h+v flip", "flip_int": "flips + brightness/contrast jitter (image only)"}[a.aug]}
    json.dump(config, open(out / "config.json", "w"), indent=1)

    start, best, best_ep, best_val, elapsed, hist = 1, -1.0, 0, None, 0.0, []
    ckpt = out / "last.pth"
    if a.resume and ckpt.exists():
        s = torch.load(ckpt, map_location=dev, weights_only=True)
        model.load_state_dict(s["model"]); opt.load_state_dict(s["opt"]); sched.load_state_dict(s["sched"])
        if scaler and s.get("scaler"):
            scaler.load_state_dict(s["scaler"])
        start, best, best_ep, best_val, elapsed = s["epoch"] + 1, s["best"], s["best_ep"], s["best_val"], s["elapsed"]
        hist = [[r[0]] + [float(v) for v in r[1:]] for r in list(csv.reader(open(out / "history.csv")))[1:]]
        print(f"resumed from epoch {s['epoch']} (best so far epoch {best_ep}, val mIoU {best:.4f})")

    t0 = time.time() - elapsed
    nan = float("nan")
    with open(out / "history.csv", "a" if start > 1 else "w", newline="") as fh:
        w = csv.writer(fh)
        if start == 1:
            w.writerow(COLS)
        for ep in range(start, a.epochs + 1):
            te = time.time()
            trl, trce, trm = run_epoch(model, dl_tr, crit, dev, opt, scaler, amp)
            val, vace, vam = run_epoch(model, dl_va, crit, dev, amp=amp)
            sched.step()
            g = lambda d, k: d[k] if d[k] is not None else nan
            row = [a.run_id, ep, trl, val, trce, vace, g(trm, "mIoU"), g(vam, "mIoU"), g(vam, "IoU_oil_spill"),
                   g(vam, "IoU_ship"), g(vam, "IoU_look_alike"), time.time() - te]
            w.writerow(row); fh.flush(); hist.append(row)
            score = g(vam, "mIoU")
            if score > best:  # checkpoint rule: highest VALIDATION mIoU
                best, best_ep, best_val = score, ep, vam
                torch.save(model.state_dict(), out / "best_model.pth")
            torch.save({"epoch": ep, "model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "scaler": scaler.state_dict() if scaler else None, "best": best, "best_ep": best_ep,
                        "best_val": best_val, "elapsed": time.time() - t0}, ckpt)  # survives a crash / timeout
            print(f"ep {ep:3d} | train {trl:.4f} val {val:.4f} | mIoU tr {g(trm,'mIoU'):.3f} va {score:.3f} | "
                  f"oil {row[8]:.3f} ship {row[9]:.3f} | {row[-1]:.0f}s" + ("  *best" if best_ep == ep else ""))

    total = time.time() - t0
    json.dump({"run_id": a.run_id, "selection_rule": "best validation mIoU", "best_epoch": best_ep,
               "train_time_s": total, "val": best_val}, open(out / "metrics.json", "w"), indent=1)

    arr = np.array([r[1:] for r in hist], dtype=float)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(arr[:, 0], arr[:, 1], label="train"); ax[0].plot(arr[:, 0], arr[:, 2], label="val"); ax[0].set_title("training objective")
    ax[1].plot(arr[:, 0], arr[:, 3], label="train"); ax[1].plot(arr[:, 0], arr[:, 4], label="val"); ax[1].set_title("plain CE (comparable across runs)")
    ax[2].plot(arr[:, 0], arr[:, 5], label="train mIoU"); ax[2].plot(arr[:, 0], arr[:, 6], label="val mIoU")
    ax[2].plot(arr[:, 0], arr[:, 7], "--", label="val oil IoU"); ax[2].plot(arr[:, 0], arr[:, 8], "--", label="val ship IoU"); ax[2].set_title("IoU")
    for x in ax:
        x.set_xlabel("epoch"); x.legend(); x.axvline(best_ep, color="gray", ls=":")
    fig.suptitle(a.run_id); fig.tight_layout(); fig.savefig(out / "curves.png", dpi=140); plt.close(fig)
    print(f"[{a.run_id}] done in {total/60:.1f} min | best epoch {best_ep} val mIoU {best:.4f} | outputs in {out}")

## Evaluation
`evaluate.py`

In [ ]:
"""Final evaluation of a finished run (confusion matrix, prediction panels, failure cases).

Defaults to the VALIDATION split. The held-out test set is only used with --final, which you
run ONCE per run after all experiments and checkpoint choices are frozen.

    python evaluate.py --data_root D --run_id U1            # validation
    python evaluate.py --data_root D --run_id U1 --final     # test, final numbers
"""
import argparse
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch



def eval_parser():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_root", required=True)
    ap.add_argument("--run_id", required=True)
    ap.add_argument("--final", action="store_true", help="evaluate on the held-out TEST set")
    ap.add_argument("--out", default="results")
    ap.add_argument("--splits", default="splits.json")
    ap.add_argument("--n_panels", type=int, default=6)
    return ap


def run_eval(a):
    run = Path(a.out) / a.run_id
    cfg = json.load(open(run / "config.json"))
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    colors = [tuple(c) for c in cfg["colors"]]
    size_hw, norm = (cfg["height"], cfg["width"]), cfg["norm"]
    found = discover(a.data_root)
    split = json.load(open(a.splits))
    if a.final:
        items, tag = find_pairs(*found["test"], prefix="test_")[0], "test"
        assert {i[0] for i in items} == set(split["test"]), "test IDs changed since the split was made"
    else:
        items, tag = select(find_pairs(*found["train"])[0], split["val"]), "val"
    print(f"evaluating {a.run_id} on {tag}: {len(items)} images")

    model = UNet(3, 5, cfg["base"], cfg["depth"]).to(dev)
    model.load_state_dict(torch.load(run / "best_model.pth", map_location=dev, weights_only=True))
    model.eval()
    ds = OilSpillDataset(items, size_hw, "none", colors, norm, return_id=True)
    cm = torch.zeros(5, 5, dtype=torch.long, device=dev)
    per_img = []  # (oil IoU or nan, missed-ship flag, index)
    with torch.no_grad():
        for i in range(len(ds)):
            x, y, _ = ds[i]
            pred = model(x[None].to(dev)).argmax(1)[0]
            y = y.to(dev)
            cm += confusion(pred, y)
            tp = ((pred == 1) & (y == 1)).sum().item(); un = ((pred == 1) | (y == 1)).sum().item()
            per_img.append((tp / un if (y == 1).any() else np.nan, bool((y == 3).any() and not (pred == 3).any()), i))
    m = summarize(cm.cpu().numpy())
    print(json.dumps(m, indent=1))

    mp = run / "metrics.json"
    allm = json.load(open(mp))
    allm[tag] = m
    json.dump(allm, open(mp, "w"), indent=1)

    c = cm.cpu().numpy().astype(float)
    norm_cm = c / np.maximum(c.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.imshow(norm_cm, cmap="Blues", vmin=0, vmax=1)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f"{norm_cm[i, j]:.2f}", ha="center", va="center", color="w" if norm_cm[i, j] > .5 else "k", fontsize=8)
    ax.set_xticks(range(5)); ax.set_xticklabels(NAMES, rotation=30); ax.set_yticks(range(5)); ax.set_yticklabels(NAMES)
    ax.set_xlabel("predicted"); ax.set_ylabel("ground truth"); ax.set_title(f"{a.run_id} {tag}: row-normalised confusion")
    fig.tight_layout(); fig.savefig(run / "confusion_matrix.png", dpi=150); plt.close(fig)

    def panel(idxs, fname, title):
        if not idxs:
            return
        fig, axs = plt.subplots(len(idxs), 3, figsize=(12, 2.6 * len(idxs)), squeeze=False)
        for r, i in enumerate(idxs):
            x, y, sid = ds[i]
            with torch.no_grad():
                p = model(x[None].to(dev)).argmax(1)[0].cpu().numpy()
            for k, (im, t) in enumerate([(denorm(x, norm), f"input {sid}"), (colorize(y.numpy(), colors), "ground truth"),
                                         (colorize(p, colors), "prediction")]):
                axs[r, k].imshow(im); axs[r, k].set_title(t, fontsize=8); axs[r, k].axis("off")
        fig.suptitle(title); fig.tight_layout(); fig.savefig(run / fname, dpi=130); plt.close(fig)

    rnd = list(range(len(ds)))
    random.Random(0).shuffle(rnd)
    panel(rnd[: a.n_panels], "predictions.png", f"{a.run_id} {tag}: random samples")
    worst_oil = [i for s, _, i in sorted([t for t in per_img if not np.isnan(t[0])])[: a.n_panels // 2 + 1]]
    missed = [i for _, f, i in per_img if f][: a.n_panels // 2]
    panel(list(dict.fromkeys(worst_oil + missed)), "failure_cases.png",
          f"{a.run_id} {tag}: lowest oil IoU (oil vs look-alike) + missed ships")
    print(f"wrote confusion_matrix.png, predictions.png, failure_cases.png and metrics.json[{tag}] in {run}")

## Comparison table
`compare.py`

In [ ]:
"""Build the comparison table over every run in results/ (U-Net runs and Omar's DeepLab runs too,
as long as each run folder follows the shared contract: config.json + metrics.json).

    python compare.py --out results

Writes results/comparison.csv and results/comparison.md. A run is reported on the TEST split if
you have run evaluate.py --final for it, otherwise on validation (flagged in the 'Split' column).
"""
import argparse
import csv
import json
from pathlib import Path

NAMES = ["sea", "oil_spill", "look_alike", "ship", "land"]
HEAD = ["Run", "Loss", "Aug", "Input", "Best ep", "Time (min)", "Split", "mIoU"] + \
       [f"IoU {n}" for n in NAMES] + ["Dice oil", "Dice ship", "Pixel acc"]


def _f(v):
    return "n/a" if v is None else f"{v:.4f}"


def make_table(out="results"):
    rows = []
    for run in sorted(p for p in Path(out).iterdir() if (p / "metrics.json").exists() and (p / "config.json").exists()):
        cfg, met = json.load(open(run / "config.json")), json.load(open(run / "metrics.json"))
        split = "test" if "test" in met else "val"
        m = met[split]
        size = f"{cfg.get('height', cfg.get('size', '?'))}x{cfg.get('width', cfg.get('size', '?'))}"
        rows.append([run.name, cfg.get("loss", "?"), cfg.get("aug", "?"), size, met.get("best_epoch"),
                     f"{met.get('train_time_s', 0) / 60:.1f}", split, _f(m.get("mIoU"))] +
                    [_f(m.get(f"IoU_{n}")) for n in NAMES] + [_f(m.get("Dice_oil_spill")), _f(m.get("Dice_ship")),
                                                              _f(m.get("pixel_acc"))])
    if not rows:
        print("no finished runs found in", out)
        return rows
    with open(Path(out) / "comparison.csv", "w", newline="") as fh:
        w = csv.writer(fh); w.writerow(HEAD); w.writerows(rows)
    md = ["| " + " | ".join(HEAD) + " |", "|" + "---|" * len(HEAD)] + ["| " + " | ".join(map(str, r)) + " |" for r in rows]
    (Path(out) / "comparison.md").write_text("\n".join(md) + "\n", encoding="utf-8")
    print("\n".join(md))
    if any(r[6] == "val" for r in rows):
        print("\nNOTE: rows marked 'val' are validation numbers. Final report numbers must come from evaluate.py --final (test).")
    return rows

## Step 1: Audit the dataset
Check exact mask colours, sizes, class balance, and that all five classes appear. Read the printout before trusting the class map.

In [ ]:
run_audit(audit_parser().parse_args(["--data_root", DATA_ROOT, "--out", "audit"]))
show("audit/class_frequency.png", "audit/sample_grid.png")

## Step 2: Experiments (U-Net)
U1 vs U2 isolates the loss. Optional ablations split U2 into weights-only and Dice-only.

In [ ]:
COMMON = ["--data_root", DATA_ROOT, "--epochs", str(EPOCHS), "--height", str(H), "--width", str(W), "--base", str(BASE),
          "--workers", str(WORKERS), "--resume"]   # --resume: a re-run continues from last.pth instead of starting over

# U1: baseline. Plain cross-entropy.
run_train(train_parser().parse_args(COMMON + ["--run_id", "U1", "--loss", "ce"]))
# U2: class-weighted cross-entropy + soft Dice. Same data/seed/size/epochs/lr; only the loss differs.
run_train(train_parser().parse_args(COMMON + ["--run_id", "U2", "--loss", "wce_dice"]))
if RUN_ABLATIONS:  # decompose U2: which part of the loss helped?
    run_train(train_parser().parse_args(COMMON + ["--run_id", "U2a_weights_only", "--loss", "wce"]))
    run_train(train_parser().parse_args(COMMON + ["--run_id", "U2b_dice_only", "--loss", "ce_dice"]))
RUNS = [r for r in ["U1", "U2", "U2a_weights_only", "U2b_dice_only"] if os.path.exists(f"results/{r}/metrics.json")]

## Step 3: Training curves
Compare train vs validation curves (not one final score) when discussing over/underfitting.

In [ ]:
for r in RUNS:
    show(f"results/{r}/curves.png")

## Step 4: Evaluation
Confusion matrix, prediction panels, failure cases (oil vs look-alike, missed ships).

In [ ]:
# Validation numbers/figures are always produced. The held-out TEST set is only touched when FINAL = True.
for r in RUNS:
    run_eval(eval_parser().parse_args(["--data_root", DATA_ROOT, "--run_id", r] + (["--final"] if FINAL else [])))
    show(f"results/{r}/confusion_matrix.png", f"results/{r}/predictions.png", f"results/{r}/failure_cases.png")

## Step 5: Comparison table
Omar's DeepLabV3+ runs appear here too if their folders (config.json + metrics.json) are copied into `results/`.

In [ ]:
rows = make_table("results")
print()
print("Report numbers must come from the TEST split (FINAL = True)." if not FINAL else "Table above is on the held-out test split.")